In [ ]:
import cv2
import numpy as np
from scipy import ndimage
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as ssim

In [ ]:
# #phương pháp bicubic"
# nameImg = "1"
# img = cv2.imread(f"{nameImg}.jpg")

# # resize bằng bicubic
# bicubic = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

# img1 = bicubic
# cv2.imwrite(f"{nameImg}_bicubic.jpg", img1)

In [ ]:
# # EIAAF
# gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)

# # tính gradient
# gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
# gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)

# mag = cv2.magnitude(gx, gy)
# dir = cv2.phase(gx, gy)

# # mờ theo hướng cạnh
# blur1 = cv2.GaussianBlur(img1, (5, 5), 0.5)
# mask = cv2.normalize(mag, None, 0, 1, cv2.NORM_MINMAX)

# # trộn
# mask = np.stack([mask] * 3, axis=-1)
# jaggy_suppressed = img1*(1-mask) + blur1*mask
# jaggy_suppressed = jaggy_suppressed.astype(np.uint8)
# img2 = jaggy_suppressed
# cv2.imwrite(f"{nameImg}_EIAAF.jpg", img2)

In [ ]:
# EDUMS
def tinh_gradient(anh):
    sobel_x = np.array([[-1, 0, 1],
                        [-2, 0, 2],
                        [-1, 0, 1]])
    sobel_y = np.array([[-1, -2, -1],
                        [ 0,  0,  0],
                        [ 1,  2,  1]])
    gx = ndimage.convolve(anh, sobel_x, mode='reflect')
    gy = ndimage.convolve(anh, sobel_y, mode='reflect')
    do_lon = np.sqrt(gx**2 + gy**2)
    huong = np.arctan2(gy, gx)
    return do_lon, huong, gx, gy

def rgb_to_ycbcr_fullrange(rgb):
    # rgb: float in 0..255
    R = rgb[...,0]; G = rgb[...,1]; B = rgb[...,2]
    Y  =  0.29900 * R + 0.58700 * G + 0.11400 * B
    Cb = 128.0 + (-0.168736 * R - 0.331264 * G + 0.5 * B)
    Cr = 128.0 + (0.5 * R - 0.418688 * G - 0.081312 * B)
    return Y, Cb, Cr

def ycbcr_to_rgb_fullrange(Y, Cb, Cr):
    cb = Cb - 128.0
    cr = Cr - 128.0
    R = Y + 1.40200 * cr
    G = Y - 0.344136 * cb - 0.714136 * cr
    B = Y + 1.77200 * cb
    return np.stack([R, G, B], axis=-1)

def edums_nhanh_ycbcr(anh, alpha=1.5, sigma=2.0, bgr=False, verbose=False):
    """
    Sharpen chỉ trên kênh Y (YCbCr full-range), giữ Cb/Cr nguyên vẹn.
    Trả về ảnh cùng dtype/scale như input. Nếu bgr=True thì input/output là BGR (cv2).
    Nếu verbose=True: in thông tin debug.
    """
    orig_dtype = anh.dtype
    arr = anh.astype(np.float64)

    # detect float 0..1 scaling
    scaled_from_0_1 = False
    if np.issubdtype(orig_dtype, np.floating) and arr.max() <= 1.0 + 1e-12:
        arr = arr * 255.0
        scaled_from_0_1 = True

    has_alpha = (arr.ndim == 3 and arr.shape[2] == 4)
    alpha_chan = None
    if has_alpha:
        alpha_chan = arr[...,3].copy()
        arr = arr[...,:3]

    if verbose:
        print("DEBUG: arr.shape", arr.shape, "dtype", orig_dtype, "scaled_from_0_1", scaled_from_0_1)

    # If grayscale input (HxW)
    if arr.ndim == 2:
        Y = arr
        do_lon, huong, gx, gy = tinh_gradient(Y)
        max_m = np.max(do_lon)
        do_lon_norm = do_lon / (max_m + 1e-10)
        w = np.clip(do_lon_norm, 0.0, 1.0)
        Y_blur = gaussian_filter(Y, sigma=sigma)
        mask = Y - Y_blur
        Y_sharp = np.clip(Y + alpha * mask * w, 0.0, 255.0)
        out = Y_sharp
    else:
        rgb = arr.copy()
        if bgr:
            rgb = rgb[..., ::-1]  # BGR -> RGB for processing

        # Convert to YCbCr (works with 0..255 float)
        Y, Cb, Cr = rgb_to_ycbcr_fullrange(rgb)

        # compute edge weight on Y
        do_lon, huong, gx, gy = tinh_gradient(Y)
        max_m = np.max(do_lon)
        do_lon_norm = do_lon / (max_m + 1e-10)
        w = np.clip(do_lon_norm, 0.0, 1.0)

        Y_blur = gaussian_filter(Y, sigma=sigma)
        mask = Y - Y_blur
        Y_sharp = np.clip(Y + alpha * mask * w, 0.0, 255.0)

        # Reconstruct RGB from sharpened Y and original Cb/Cr
        out_rgb = ycbcr_to_rgb_fullrange(Y_sharp, Cb, Cr)
        # Clip and ensure shape HxWx3
        out_rgb = np.clip(out_rgb, 0.0, 255.0)

        if bgr:
            out_rgb = out_rgb[..., ::-1]

        out = out_rgb

    # reattach alpha channel if present
    if has_alpha:
        alpha_chan_clipped = np.clip(alpha_chan, 0.0, 255.0)
        out = np.concatenate([out, alpha_chan_clipped[..., np.newaxis]], axis=2)

    # cast back to original dtype/scale
    if scaled_from_0_1:
        out = out / 255.0
        out = out.astype(orig_dtype)
    else:
        if np.issubdtype(orig_dtype, np.integer):
            out = np.clip(out, 0, 255).round().astype(orig_dtype)
        else:
            out = out.astype(orig_dtype)

    if verbose:
        print("DEBUG: out.shape", out.shape, "out.dtype", out.dtype,
              "min/max:", out.min(), out.max())
        if out.ndim == 3:
            print("DEBUG: channel means:", [out[...,c].mean() for c in range(out.shape[2])])

    return out

In [ ]:
# # in ra ảnh sau khi dùng EDUMS
# img3 = edums_nhanh_ycbcr(img2, alpha=1.5, sigma=1.5, bgr=True, verbose=True)
# cv2.imwrite(f'{nameImg}_edums.png', img3)

In [ ]:
#Khử rung EICL
def phat_hien_canh_sobel(anh_kenh, scale=1.0):
    """
    Phát hiện cạnh bằng Sobel trên một kênh
    """
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]) * scale
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]) * scale
    
    gx = ndimage.convolve(anh_kenh, sobel_x, mode='reflect')
    gy = ndimage.convolve(anh_kenh, sobel_y, mode='reflect')
    
    edge_magnitude = np.sqrt(gx**2 + gy**2)
    edge_magnitude = edge_magnitude / (edge_magnitude.max() + 1e-10)
    
    return edge_magnitude


def tao_ban_do_trong_so(edge_map, nguong=0.1, sigma_expand=2.0):
    """
    Tạo bản đồ trọng số từ edge map
    Vùng gần cạnh có trọng số cao hơn (cần loại bỏ ringing nhiều hơn)
    """
    # Tạo binary edge map
    binary_edges = (edge_map > nguong).astype(float)
    
    # Mở rộng vùng cạnh bằng Gaussian
    weight_map = gaussian_filter(binary_edges, sigma=sigma_expand)
    weight_map = np.clip(weight_map, 0.0, 1.0)
    
    return weight_map


def eicl_mot_kenh(kenh, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5):
    """
    Áp dụng EICL lên một kênh ảnh (Y, hoặc R, G, B)
    
    Args:
        kenh: Kênh ảnh (0-255)
        nguong_canh: Ngưỡng phát hiện cạnh
        sigma_smooth: Độ mạnh làm mượt
        alpha: Cường độ loại bỏ ringing (0-1)
    
    Returns:
        Kênh đã xử lý
    """
    # Phát hiện cạnh
    edge_map = phat_hien_canh_sobel(kenh)
    
    # Tạo bản đồ trọng số
    weight_map = tao_ban_do_trong_so(edge_map, nguong=nguong_canh, sigma_expand=sigma_smooth * 1.5)
    
    # Làm mượt kênh
    kenh_muot = gaussian_filter(kenh, sigma=sigma_smooth)
    
    # Trộn: vùng có ringing (gần cạnh) sẽ được làm mượt nhiều hơn
    kenh_eicl = kenh * (1.0 - alpha * weight_map) + kenh_muot * (alpha * weight_map)
    
    return kenh_eicl, edge_map, weight_map


def eicl_sau_edums(anh_edums, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5, 
                   bgr=False, verbose=False):
    """
    Áp dụng EICL để loại bỏ ringing artifacts sau khi sharpen bằng EDUMS.
    Xử lý trên không gian YCbCr, chỉ làm mượt kênh Y để giữ màu sắc.
    
    Args:
        anh_edums: Ảnh output từ EDUMS
        nguong_canh: Ngưỡng phát hiện cạnh (0.05-0.15 tốt)
        sigma_smooth: Độ mạnh làm mượt (1.0-2.0 tốt)
        alpha: Cường độ loại bỏ ringing (0.3-0.7 tốt)
        bgr: True nếu input là BGR (OpenCV)
        verbose: In thông tin debug
    
    Returns:
        Ảnh đã loại bỏ ringing, cùng dtype/scale với input
    """
    orig_dtype = anh_edums.dtype
    arr = anh_edums.astype(np.float64)
    
    # Kiểm tra scale 0-1
    scaled_from_0_1 = False
    if np.issubdtype(orig_dtype, np.floating) and arr.max() <= 1.0 + 1e-12:
        arr = arr * 255.0
        scaled_from_0_1 = True
    
    # Xử lý alpha channel
    has_alpha = (arr.ndim == 3 and arr.shape[2] == 4)
    alpha_chan = None
    if has_alpha:
        alpha_chan = arr[..., 3].copy()
        arr = arr[..., :3]
    
    if verbose:
        print(f"EICL: shape={arr.shape}, dtype={orig_dtype}, scaled_0_1={scaled_from_0_1}")
    
    # Xử lý ảnh xám
    if arr.ndim == 2:
        Y_eicl, edge_map, weight_map = eicl_mot_kenh(arr, nguong_canh, sigma_smooth, alpha)
        out = np.clip(Y_eicl, 0.0, 255.0)
        
    # Xử lý ảnh màu
    else:
        rgb = arr.copy()
        if bgr:
            rgb = rgb[..., ::-1]  # BGR -> RGB
        
        # Chuyển sang YCbCr
        Y, Cb, Cr = rgb_to_ycbcr_fullrange(rgb)
        
        # Chỉ áp dụng EICL lên kênh Y (giữ nguyên Cb, Cr)
        Y_eicl, edge_map, weight_map = eicl_mot_kenh(Y, nguong_canh, sigma_smooth, alpha)
        Y_eicl = np.clip(Y_eicl, 0.0, 255.0)
        
        # Chuyển ngược về RGB
        out_rgb = ycbcr_to_rgb_fullrange(Y_eicl, Cb, Cr)
        out_rgb = np.clip(out_rgb, 0.0, 255.0)
        
        if bgr:
            out_rgb = out_rgb[..., ::-1]  # RGB -> BGR
        
        out = out_rgb
    
    # Gắn lại alpha channel
    if has_alpha:
        out = np.concatenate([out, alpha_chan[..., np.newaxis]], axis=2)
    
    # Chuyển về dtype/scale gốc
    if scaled_from_0_1:
        out = out / 255.0
        out = out.astype(orig_dtype)
    else:
        if np.issubdtype(orig_dtype, np.integer):
            out = np.clip(out, 0, 255).round().astype(orig_dtype)
        else:
            out = out.astype(orig_dtype)
    
    if verbose:
        print(f"EICL output: shape={out.shape}, dtype={out.dtype}, range=[{out.min():.2f}, {out.max():.2f}]")
    
    return out


In [ ]:
# img4 = eicl_sau_edums(img3, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5, bgr=True, verbose=True)
# cv2.imwrite(f'{nameImg}_eicl.png', img4)

In [ ]:
# # đánh giá chất lượng ảnh
# ## chỉ sử dụng trong trường hợp 2 ảnh có cùng kích thước
# # Use the original upscaled image for comparison
# img5 = img1  # Original bicubic upscaled image
# img6 = img4  # EIAAF processed image

# # Convert img3 (single channel output) to 3-channel for comparison if needed
# # Ensure both images have the same shape for comparison
# if img5.shape != img6.shape:
# 	img6 = cv2.cvtColor(img6.astype(np.uint8), cv2.COLOR_GRAY2BGR)

# # Tính các chỉ số
# psnr_value = psnr(img5, img6)
# ssim_value = ssim(img5, img6, channel_axis=-1)
# mse_value = mse(img5, img6)
# rms_value = np.sqrt(mse_value)

# print(f"PSNR: {psnr_value}")
# print(f"SSIM: {ssim_value}")
# print(f"MSE: {mse_value}")
# print(f"RMS: {rms_value}")

In [ ]:
# so sánh thông số và ghi vào file (so sánh 2 ảnh có kích thước khác nhau)
def compare_images_2(img1_path, img2_path, nameImg, output_file='comparison_results.txt'):
    """So sánh các thông số của 2 ảnh có kích thước khác nhau và ghi vào file"""
    
    # Mở file để ghi (mode 'a' để append vào file sẵn có)
    with open(output_file, 'a', encoding='utf-8') as f:
        # Đọc ảnh
        img1 = img1_path
        img2 = img2_path
    
        # 1. So sánh kích thước gốc
        f.write(f"=== THÔNG SỐ CƠ BẢN ẢNH SỐ {nameImg}===\n")
        f.write(f"Ảnh gốc: {img1.shape} (HxWxC)\n")
        f.write(f"Ảnh qua xử lý: {img2.shape} (HxWxC)\n")
        f.write(f"Tỷ lệ kích thước: {img1.shape[0]/img2.shape[0]:.2f}x{img1.shape[1]/img2.shape[1]:.2f}\n")
        
        # 2. So sánh số lượng pixel
        pixels1 = img1.shape[0] * img1.shape[1]
        pixels2 = img2.shape[0] * img2.shape[1]
        f.write(f"\nSố pixel - Ảnh gốc: {pixels1:,} | Ảnh qua xử lý: {pixels2:,}\n")
        
        # 3. So sánh giá trị trung bình màu sắc
        f.write("\n=== THÔNG SỐ MÀU SẮC ===\n")
        mean1 = np.mean(img1, axis=(0,1))
        mean2 = np.mean(img2, axis=(0,1))
        f.write(f"Giá trị TB (BGR) - Ảnh gốc: {mean1}\n")
        f.write(f"Giá trị TB (BGR) - Ảnh qua xử lý: {mean2}\n")
        
        # 4. So sánh độ sáng
        brightness1 = np.mean(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY))
        brightness2 = np.mean(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY))
        f.write(f"Độ sáng TB - Ảnh gốc: {brightness1:.2f} | Ảnh qua xử lý: {brightness2:.2f}\n")
        
        # 5. So sánh độ tương phản
        std1 = np.std(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY))
        std2 = np.std(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY))
        f.write(f"Độ tương phản - Ảnh gốc: {std1:.2f} | Ảnh qua xử lý: {std2:.2f}\n")
        
        # 6. So sánh histogram
        f.write("\n=== SO SÁNH HISTOGRAM ===\n")
        hist1 = cv2.calcHist([img1], [0,1,2], None, [8,8,8], [0,256,0,256,0,256])
        hist2 = cv2.calcHist([img2], [0,1,2], None, [8,8,8], [0,256,0,256,0,256])
        hist1 = cv2.normalize(hist1, hist1).flatten()
        hist2 = cv2.normalize(hist2, hist2).flatten()
        hist_correlation = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)
        f.write(f"Tương quan histogram: {hist_correlation:.4f} (1 = giống nhất)\n")
        
        # 7. So sánh SSIM (cần resize về cùng kích thước)
        f.write("\n=== SO SÁNH NỘI DUNG (SSIM) ===\n")
        # Resize về kích thước nhỏ hơn để so sánh
        h = min(img1.shape[0], img2.shape[0])
        w = min(img1.shape[1], img2.shape[1])
        img1_resized = cv2.resize(img1, (w, h))
        img2_resized = cv2.resize(img2, (w, h))
        
        # Chuyển sang grayscale
        gray1 = cv2.cvtColor(img1_resized, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2_resized, cv2.COLOR_BGR2GRAY)
        
        ssim_score = ssim(gray1, gray2)
        f.write(f"SSIM Score: {ssim_score:.4f} (1 = giống hệt)\n")
        
        # 8. So sánh MSE (Mean Squared Error)
        mse = np.mean((img1_resized.astype(float) - img2_resized.astype(float)) ** 2)
        f.write(f"MSE: {mse:.2f} (0 = giống hệt)\n")
        
        # 9. Tính PSNR
        if mse > 0:
            psnr = 20 * np.log10(255.0 / np.sqrt(mse))
            f.write(f"PSNR: {psnr:.2f} dB (>40 = rất giống)\n")
        f.write("=" * 40)
        f.write("\n")
        
    print(f"✓ Kết quả đã được ghi vào file: {output_file}")

In [ ]:
# compare_images_2(img, img4, nameImg, output_file='Thong_so_co_ban.txt')

In [ ]:
#IESR
def IESR(img, iters=3):
    ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    Y = ycbcr[:,:,0]
    Cr = ycbcr[:,:,1]
    Cb = ycbcr[:,:,2]

    Y_up  = cv2.resize(Y, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    Cr_up = cv2.resize(Cr, None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST)
    Cb_up = cv2.resize(Cb, None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST)

    for _ in range(iters):
        edge = cv2.Laplacian(Y_up, cv2.CV_32F)
        Y_up = cv2.addWeighted(Y_up.astype(np.float32), 1.0,
                               edge.astype(np.float32), 0.15, 0)
        Y_up = np.clip(Y_up, 0, 255).astype(np.uint8)

    out = cv2.merge([Y_up, Cr_up, Cb_up])
    return cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)


In [ ]:
for i in range(1, 1255):
    #phương pháp bicubic"
    nameImg = f"{i}"
    img = cv2.imread(f"Dataset/{nameImg}.jpg")

    # resize bằng bicubic
    bicubic = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    img1 = bicubic
    # cv2.imwrite(f"Output/{nameImg}_bicubic.jpg", img1)

    # EIAAF
    gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)

    # tính gradient
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)

    mag = cv2.magnitude(gx, gy)
    dir = cv2.phase(gx, gy)

    # mờ theo hướng cạnh
    blur1 = cv2.GaussianBlur(img1, (5, 5), 0.5)
    mask = cv2.normalize(mag, None, 0, 1, cv2.NORM_MINMAX)

    # trộn
    mask = np.stack([mask] * 3, axis=-1)
    jaggy_suppressed = img1*(1-mask) + blur1*mask
    jaggy_suppressed = jaggy_suppressed.astype(np.uint8)
    img2 = jaggy_suppressed
    # cv2.imwrite(f"Output/{nameImg}_EIAAF.jpg", img2)

    # in ra ảnh sau khi dùng EDUMS
    img3 = edums_nhanh_ycbcr(img2, alpha=1.5, sigma=1.5, bgr=True, verbose=True)
    # cv2.imwrite(f'Output/{nameImg}_edums.png', img3)

    #Khử rung EICL
    img4 = eicl_sau_edums(img3, nguong_canh=0.08, sigma_smooth=1.2, alpha=0.5, bgr=True, verbose=True)
    # cv2.imwrite(f'Output/Use_edums/{nameImg}_eicl.png', img4)

    img5 = IESR(img)
    # cv2.imwrite(f'Output/Use_IESR/{nameImg}_iesr.png', img5)
    # so sánh thông số và ghi vào file (so sánh 2 ảnh có kích thước khác nhau)
    compare_images_2(img, img4, nameImg, output_file='Thong_so_co_ban.txt')
    compare_images_2(img, img5, nameImg, output_file='Thong_so_co_ban_2.txt')